# ANNA RL Training - Google Colab

Este notebook entrena un modelo PPO para ANNA usando la infraestructura de Google Colab.

## Requisitos
- Runtime: **GPU** (recomendado) o T4
- ~10GB de espacio en disco

## 1. Clonar repositorio

In [ ]:
import os
os.chdir('/content')

!rm -rf Odisea_Game
!git clone https://github.com/icarito/Odisea.git Odisea_Game
%cd Odisea_Game/

## 2. Instalar dependencias Python

In [ ]:
!pip install gymnasium stable-baselines3 torch numpy

## 3. Instalar Godot 3.6 Headless

In [ ]:
import os
import urllib.request
import zipfile

godot_url = "https://github.com/godotengine/godot/releases/download/3.6-stable/Godot_v3.6-stable_linux_headless.64.zip"
godot_zip = "/tmp/godot_headless.zip"
godot_dir = "/opt/godot"

os.makedirs(godot_dir, exist_ok=True)

print("Downloading Godot 3.6 headless...")
urllib.request.urlretrieve(godot_url, godot_zip)

with zipfile.ZipFile(godot_zip, 'r') as zf:
    zf.extractall(godot_dir)

godot_bin = os.path.join(godot_dir, "Godot_v3.6-stable_linux_headless.64")
os.chmod(godot_bin, 0o755)

os.environ['GODOT_BIN'] = godot_bin
os.environ['PATH'] = godot_dir + ':' + os.environ['PATH']

!{godot_bin} --version

## 4. Desactivar editor plugins (para import CI)

In [ ]:
import os
os.chdir('/content/Odisea_Game')

# Disable editor plugins (from GitHub Actions workflow)
!cp project.godot project.godot.ci.bak

with open('project.godot.ci.bak', 'r') as f:
    content = f.read()

# Replace enabled=PoolStringArray(...) with empty
import re
content = re.sub(
    r'enabled=PoolStringArray\([^)]*\)',
    'enabled=PoolStringArray( )',
    content
)

with open('project.godot', 'w') as f:
    f.write(content)

print("Editor plugins disabled for CI import.")

## 5. Importar assets de Godot (CRÍTICO)

Este paso importa todos los recursos. Puede tomar 3-5 minutos.

In [ ]:
import os
import subprocess
import signal

os.chdir('/content/Odisea_Game')
godot_bin = os.environ['GODOT_BIN']

print("Importing Godot resources (this takes 3-5 minutes)...")
print("Using editor mode import like GitHub Actions CI...")

# Run import with editor mode (same as GitHub Actions)
cmd = [
    godot_bin,
    '--path', '.',
    '-e',  # Editor mode - crucial for proper import
    '--headless',
    '--no-window',
    '--audio-driver', 'Dummy',
    '--quit-after', '300'  # Quit after 300 frames max
]

result = subprocess.run(cmd, capture_output=False, timeout=600)

print(f"\n=== Import exit code: {result.returncode} ===")
print("\n=== Checking .import folder ===")
!ls -la .import/ 2>/dev/null | head -10 || echo "No .import folder found"

## 6. Verificar importación

In [ ]:
# Check if critical files exist
import os
os.chdir('/content/Odisea_Game')

critical_files = [
    '.import/joystick_base_outline.png-1529fbc0a23b5af9e961e1a3d047aa0b.stex',
    '.import/Step_Tile_01.wav-cfe6295871f73b6f01c16fcc027c948c.sample',
]

all_ok = True
for f in critical_files:
    if os.path.exists(f):
        print(f"✅ {f}")
    else:
        print(f"❌ {f} - MISSING")
        all_ok = False

# Check total imported files
import_count = len([f for f in os.listdir('.import') if os.path.isfile(os.path.join('.import', f))])
print(f"\nTotal files in .import: {import_count}")

if all_ok:
    print("\n✅ Import verification passed!")
else:
    print("\n⚠️ Some files missing - training may fail")

## 7. Configurar variables de entorno

In [ ]:
import os
os.environ['GODOT_BIN'] = '/opt/godot/Godot_v3.6-stable_linux_headless.64'
os.environ['ANNA_RL_MODE'] = '1'
os.chdir('/content/Odisea_Game')
print(f"Working dir: {os.getcwd()}")
print(f"GODOT_BIN: {os.environ['GODOT_BIN']}")

## 8. Entrenar modelo ANNA

Ajusta `TIMESTEPS` según el tiempo disponible.

In [ ]:
TIMESTEPS = 500000  # Ajustar según tiempo disponible
CPU_THREADS = 2     # Colab free tier tiene 2 vCPUs
SEED = 42

!python3 agents/train_anna.py \
    --timesteps {TIMESTEPS} \
    --cpu-threads {CPU_THREADS} \
    --seed {SEED} \
    --model-out agents/models/anna_ppo_colab.zip \
    --tensorboard-log agents/runs/tensorboard_colab \
    --no-launch

## 9. Ver resultados

In [ ]:
import json
from pathlib import Path

meta_path = Path('agents/models/anna_ppo_colab.meta.json')
if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    print("=== Training Complete ===")
    print(f"Duration: {meta.get('duration_sec', 0):.1f} seconds")
    print(f"Timesteps: {meta.get('timesteps', 0)}")
    print(f"Model saved: {meta.get('model')}")
else:
    print("No metadata found - training may have failed")

!ls -la agents/models/

## 10. Descargar modelo

In [ ]:
from google.colab import files

model_path = 'agents/models/anna_ppo_colab.zip'
meta_path = 'agents/models/anna_ppo_colab.meta.json'

if Path(model_path).exists():
    files.download(model_path)
    files.download(meta_path)
else:
    print("Model not found!")

## 11. (Opcional) TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir agents/runs/tensorboard_colab